In [ ]:
#cleaning
import pandas as pd
import numpy as np

#step 1 : read csv file + Decide how to handle missing and inconsistent values.

df=pd.read_csv("countries_ff.csv")
df = df.drop_duplicates()                 #remove duplicate rows
df =df.drop_duplicates(subset=["name"])   #it will remove the duplicate rows based on the "name" column
df["name"]=df["name"].str.strip()         #it will remove the leading and trailing spaces from the "name" column

#for the none values
for column in ["borders", "currencies", "languages"]:
    df[column]=df[column].fillna("None")
    df[column]=df[column].astype(str).str.strip(", ")        #remove leading and trailing commas and spaces from the columns in the for loop.
    df.loc[df[column] == "", column] = "None"                #it will replace the empty strings in the columns with "None"


df["region"] = df["region"].fillna("Unknown")         #it will replace the missing values in the region column with the string "Unknown"
df["subregion"] = df["subregion"].fillna("Unknown")   #it will replace the missing values in the subregion column with the string "Unknown"


df["area"] = pd.to_numeric(df["area"], errors="coerce")    #it will convert the values in the area column to numeric data type, and any values that cannot be converted will be replaced by NaN

df.loc[df["area"] == 0, "area"] = np.nan    #there is no way the area = 0, so instead of 0 we will replace it with NaN


df["area"] = df["area"].fillna(df["area"].median())    #it will fill the missing values in this column using the impute method, so they will be filled with the median value
#I chose the median because it does not get affected by outliers (here like the big countries)


for column in ["area"]:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[column + "_outlier"] = (df[column] < lower_bound) | (df[column] > upper_bound)
    #it creates a new column and gives it a name, it stores True or False for each country we have (True is outlier, False is normal)

df.to_csv("countries_cleaned.csv", index=False)
print("Saved countries_cleaned.csv with", len(df), "rows")